# KYC Synthetic Dataset — Postgres Load & Setup

**Purpose:** load `KYC_Synthetic_Dataset.csv` (2,515,262 rows) into PostgreSQL 18.1, database `study`,
schema `kyc`, so the Task 1 funnel analysis can be done in SQL.

**Design:** two-layer load, which keeps the work auditable.

| Layer | Table | Contents |
|---|---|---|
| Landing | `kyc.kyc_raw` | Verbatim CSV. Every column `TEXT`, no constraints, no cleaning. |
| Reference | `kyc.ref_state` | USPS code ↔ full-name lookup. |
| Analytical | `kyc.kyc_users` | Typed, cleaned, constrained, indexed. Analyse from here. |

Nothing is cleaned on the way in. Every transformation happens in SQL between `kyc_raw` and
`kyc_users`, so any cleaning decision can be re-checked against the original later.

Run order: top to bottom. All cells are idempotent — re-running rebuilds cleanly.

Credentials come from `.env` (git-ignored, chmod 600). No secret appears in this notebook or in `AUDIT_LOG.md`.

> **Server note.** The database on `localhost:5432` is a **Docker container** (`postgres`, PostgreSQL
> 18.1 on Debian/aarch64), *not* the macOS EDB PostgreSQL 17 install at `/Library/PostgreSQL/17`.
> Those v17 client binaries are what a bare `psql` resolves to and they will refuse to `pg_dump` an
> 18.1 server. Use this notebook's psycopg connection, or the container's own `psql`.

## 0 — Configuration and connection

In [1]:
import os, csv, sys, time, json, textwrap, datetime as dt
from pathlib import Path
from collections import Counter

import psycopg
import pandas as pd
from dotenv import load_dotenv

PROJECT = Path('/Users/rudransh/d_drive/GITHUB/Brightmoney')
CSV_PATH = PROJECT / 'KYC_Synthetic_Dataset.csv'
AUDIT_MD = PROJECT / 'AUDIT_LOG.md'

load_dotenv(PROJECT / '.env')

# .env supplies PGPASSWORD only; everything else defaults to the Docker postgres container
# published on localhost:5432. Override any of these in .env if the setup changes.
CONN = dict(
    host     = os.getenv('PGHOST', 'localhost'),
    port     = int(os.getenv('PGPORT', 5432)),
    user     = os.getenv('PGUSER', 'postgres'),
    password = os.getenv('PGPASSWORD'),
    dbname   = os.getenv('PGDATABASE', 'study'),
)
assert CONN['password'], 'PGPASSWORD not found in .env'

SCHEMA = 'kyc'
RUN_LOG = []   # (step, detail, seconds) collected for the audit log in the last cell

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

print('CSV      :', CSV_PATH, f'({CSV_PATH.stat().st_size/1e6:.1f} MB)')
print('Target   :', f"{CONN['user']}@{CONN['host']}:{CONN['port']}/{CONN['dbname']}", '| schema', SCHEMA)
print('Password : loaded from .env (not shown)')

CSV      : /Users/rudransh/d_drive/GITHUB/Brightmoney/KYC_Synthetic_Dataset.csv (273.9 MB)
Target   : postgres@localhost:5432/study | schema kyc
Password : loaded from .env (not shown)


In [2]:
def connect():
    'Open a fresh autocommit connection.'
    return psycopg.connect(**CONN, autocommit=True)

def run(sql, params=None, note=None):
    'Execute a statement (or script). Returns elapsed seconds.'
    t0 = time.time()
    with connect() as c, c.cursor() as cur:
        cur.execute(sql, params)
    el = time.time() - t0
    if note:
        RUN_LOG.append((note, '', el))
        print(f'{note}  ({el:.2f}s)')
    return el

def q(sql, params=None):
    'Run a query, return a DataFrame.'
    with connect() as c, c.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

def scalar(sql, params=None):
    with connect() as c, c.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchone()[0]

# Preflight: prove we are on the server and database we think we are.
print(q('''
    select current_database() as db,
           current_user       as usr,
           version()          as server
''').to_string(index=False))

   db      usr                                                                                                                   server
study postgres PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [3]:
# Safety check. `study` is a pre-existing, shared database, so this notebook must never clobber
# something it did not create. It owns exactly three tables; anything else in the `kyc` schema
# belongs to someone else and stops the run.
OWNED_TABLES = {'kyc_raw', 'ref_state', 'kyc_users'}
ALLOW_FOREIGN_OVERWRITE = False   # set True only to deliberately overwrite unrecognised tables

schema_exists = scalar('select exists (select 1 from information_schema.schemata where schema_name = %s)', (SCHEMA,))
existing = set(q('''
    select table_name from information_schema.tables where table_schema = %s
''', (SCHEMA,))['table_name']) if schema_exists else set()

foreign = existing - OWNED_TABLES

print(f'schema {SCHEMA!r} exists : {schema_exists}')
print(f'tables present         : {sorted(existing) or "none"}')
print(f'owned by this notebook : {sorted(existing & OWNED_TABLES) or "none"}  (will be rebuilt)')
print(f'not recognised         : {sorted(foreign) or "none"}')

if foreign and not ALLOW_FOREIGN_OVERWRITE:
    raise SystemExit(
        f'STOP: {SCHEMA} contains tables this notebook did not create: {sorted(foreign)}.\n'
        f'Nothing has been modified. Review them, then set ALLOW_FOREIGN_OVERWRITE = True to proceed.')

print('\nSafe to proceed.' if not foreign else '\nProceeding despite unrecognised tables (override set).')
print('Note: kyc_raw / ref_state / kyc_users are DROPped and rebuilt by the cells below.')

schema 'kyc' exists : True
tables present         : ['kyc_raw', 'kyc_users', 'ref_state']
owned by this notebook : ['kyc_raw', 'kyc_users', 'ref_state']  (will be rebuilt)
not recognised         : none

Safe to proceed.
Note: kyc_raw / ref_state / kyc_users are DROPped and rebuilt by the cells below.


## 1 — Profile the CSV before loading

Done first, in Python, so the schema is chosen from what the file actually contains rather than from
what the brief says it contains. It also gives an independent record count to reconcile the load against.

This reads the file with a real CSV parser — the file contains quoted fields with embedded commas
**and embedded newlines**, so `wc -l` overstates the record count.

In [4]:
csv.field_size_limit(10**9)
t0 = time.time()

LOWCARD = ['state','onboarded_bank_partner','kyc_source','idology_result','lexis_nexis_result',
           'persona_idv_result','persona_ssn_result','acro_result','manual_review_result','overall_kyc_status']

with open(CSV_PATH, newline='', encoding='utf-8', errors='replace') as f:
    rdr = csv.reader(f)
    header = next(rdr)
    ncol = len(header)
    n_rows = 0
    ragged = Counter()
    empties = Counter()
    vals = {c: Counter() for c in LOWCARD}
    uids = set()
    dup_uid = 0
    multiline_comments = 0
    for row in rdr:
        n_rows += 1
        if len(row) != ncol:
            ragged[len(row)] += 1
            continue
        d = dict(zip(header, row))
        for c, v in d.items():
            if v == '':
                empties[c] += 1
        for c in LOWCARD:
            vals[c][d[c]] += 1
        u = d['Fabricated_UID']
        if u in uids:
            dup_uid += 1
        else:
            uids.add(u)
        rc = d['reviewer_comment']
        if rc and ('\n' in rc or '\r' in rc):
            multiline_comments += 1

CSV_RECORD_COUNT = n_rows          # reconciliation baseline for the COPY
elapsed = time.time() - t0
RUN_LOG.append(('Profiled CSV', f'{CSV_RECORD_COUNT:,} records, {ncol} columns', elapsed))

print('columns          :', ncol)
print('records (parsed) :', f'{CSV_RECORD_COUNT:,}')
print('ragged rows      :', dict(ragged) or 'none')
print('distinct UIDs    :', f'{len(uids):,}', '| duplicates:', dup_uid)
print('multiline comments:', multiline_comments)
print(f'({elapsed:.1f}s)')

columns          : 17
records (parsed) : 2,515,262
ragged rows      : none
distinct UIDs    : 2,515,262 | duplicates: 0
multiline comments: 8
(9.6s)


In [5]:
# Null (empty-string) rate per column — drives which columns are nullable.
null_profile = (pd.DataFrame({'column': header,
                              'empty': [empties[c] for c in header]})
                  .assign(pct=lambda d: (d['empty'] / CSV_RECORD_COUNT * 100).round(2)))
print(null_profile.to_string(index=False))

                column   empty   pct
        Fabricated_UID       0  0.00
      Fabricated_Email       0  0.00
        Fabricated_SSN       0  0.00
         enrolled_date       0  0.00
                   dob     868  0.03
                 state     878  0.03
onboarded_bank_partner   12857  0.51
 checking_bank_partner  101091  4.02
            kyc_source  197314  7.84
        idology_result   94184  3.74
    lexis_nexis_result 2238263 88.99
    persona_idv_result 2512667 99.90
    persona_ssn_result 2504529 99.57
           acro_result 2401035 95.46
  manual_review_result 2440834 97.04
      reviewer_comment 2440834 97.04
    overall_kyc_status       0  0.00


In [6]:
# Distinct values of the categorical columns — confirms the domain of each result column
# and surfaces anything the brief did not mention.
for c in LOWCARD:
    v = vals[c]
    top = ', '.join(f'{k or "<empty>"}={n:,}' for k, n in v.most_common(8))
    print(f'{c:24s} distinct={len(v):<5d} {top}')
    if len(v) > 8:
        print(f'{"":24s} ... {len(v)-8} more')

state                    distinct=103   TX=272,789, FL=194,141, CA=193,994, GA=146,404, NY=111,275, NC=102,870, PA=90,392, IL=85,479
                         ... 95 more
onboarded_bank_partner   distinct=4     Bank A=2,150,325, Bank B=328,741, Bank C=23,339, <empty>=12,857
kyc_source               distinct=9     IDOLOGY=2,078,101, <empty>=197,314, ProviderA_Lexis_Nexis=149,364, ACRO=44,563, MANUAL_REVIEW=22,156, ProviderB_Lexis_Nexis=15,998, PERSONA_SSN=5,146, PERSONA_IDV=2,513
                         ... 1 more
idology_result           distinct=3     PASS=2,132,944, FAIL=288,134, <empty>=94,184
lexis_nexis_result       distinct=3     <empty>=2,238,263, PASS=183,408, FAIL=93,591
persona_idv_result       distinct=2     <empty>=2,512,667, PASS=2,595
persona_ssn_result       distinct=2     <empty>=2,504,529, PASS=10,733
acro_result              distinct=3     <empty>=2,401,035, FAIL=63,038, PASS=51,189
manual_review_result     distinct=3     <empty>=2,440,834, FAIL=50,396, PASS=24,032
ov

In [7]:
# QUIRK CHECK: the `state` column mixes USPS 2-letter codes with full state names.
USPS = {'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA',
        'ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
        'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC','PR','VI','GU','AS','MP'}

non_code = {k: n for k, n in vals['state'].items() if k and k not in USPS}
print(f'state values that are NOT a USPS code: {len(non_code)} distinct, '
      f'{sum(non_code.values()):,} rows ({sum(non_code.values())/CSV_RECORD_COUNT*100:.2f}%)')
for k, n in sorted(non_code.items(), key=lambda x: -x[1])[:10]:
    print(f'   {k:24s} {n:>8,}')
print('\nSame state can appear both ways, e.g. OH =', f"{vals['state'].get('OH',0):,}",
      'and Ohio =', f"{vals['state'].get('Ohio',0):,}")
print('=> normalised to a single state_code in kyc_users (step 5).')

state values that are NOT a USPS code: 51 distinct, 189,795 rows (7.55%)
   Ohio                       26,219
   Virginia                   17,328
   Alabama                    15,894
   Louisiana                  15,541
   South Carolina             14,698
   Indiana                    14,390
   Missouri                   12,596
   Mississippi                10,666
   Washington                  9,247
   Oklahoma                    7,939

Same state can appear both ways, e.g. OH = 76,705 and Ohio = 26,219
=> normalised to a single state_code in kyc_users (step 5).


## 2 — Roles

Three NOLOGIN group roles, granted to the working user. `kyc_ro` is the safe default for ad-hoc
analysis; `kyc_rw` and `kyc_owner` exist so privileges can be handed out without sharing the superuser.

In [8]:
run('''
do $$
begin
  if not exists (select 1 from pg_roles where rolname = 'kyc_owner') then create role kyc_owner nologin; end if;
  if not exists (select 1 from pg_roles where rolname = 'kyc_rw')    then create role kyc_rw    nologin; end if;
  if not exists (select 1 from pg_roles where rolname = 'kyc_ro')    then create role kyc_ro    nologin; end if;
end $$;
''', note='Created roles kyc_owner / kyc_rw / kyc_ro')

print(q("select rolname, rolcanlogin as can_login from pg_roles where rolname like 'kyc%' order by rolname").to_string(index=False))

Created roles kyc_owner / kyc_rw / kyc_ro  (0.01s)
  rolname  can_login
kyc_owner      False
   kyc_ro      False
   kyc_rw      False


## 3 — Schema, landing table, state reference

`kyc_raw` is deliberately all `TEXT` with no constraints: the COPY cannot fail on a bad cast, so a
malformed value becomes a visible data-quality finding rather than an aborted load.

In [9]:
run(f'create schema if not exists {SCHEMA} authorization current_user;', note=f'Created schema {SCHEMA}')

run(f'''
drop table if exists {SCHEMA}.kyc_raw;
create table {SCHEMA}.kyc_raw (
    fabricated_uid          text,
    fabricated_email        text,
    fabricated_ssn          text,
    enrolled_date           text,
    dob                     text,
    state                   text,
    onboarded_bank_partner  text,
    checking_bank_partner   text,
    kyc_source              text,
    idology_result          text,
    lexis_nexis_result      text,
    persona_idv_result      text,
    persona_ssn_result      text,
    acro_result             text,
    manual_review_result    text,
    reviewer_comment        text,
    overall_kyc_status      text
);
comment on table {SCHEMA}.kyc_raw is
  'Verbatim landing table for KYC_Synthetic_Dataset.csv. All TEXT, no constraints. Analyse from kyc.kyc_users instead.';
''', note='Created kyc.kyc_raw (landing table)')

Created schema kyc  (0.01s)
Created kyc.kyc_raw (landing table)  (0.03s)


0.03080916404724121

In [10]:
run(f'''
drop table if exists {SCHEMA}.ref_state;
create table {SCHEMA}.ref_state (
    state_code text primary key,
    state_name text not null unique
);
insert into {SCHEMA}.ref_state (state_code, state_name) values
 ('AL','Alabama'),('AK','Alaska'),('AZ','Arizona'),('AR','Arkansas'),('CA','California'),
 ('CO','Colorado'),('CT','Connecticut'),('DE','Delaware'),('FL','Florida'),('GA','Georgia'),
 ('HI','Hawaii'),('ID','Idaho'),('IL','Illinois'),('IN','Indiana'),('IA','Iowa'),
 ('KS','Kansas'),('KY','Kentucky'),('LA','Louisiana'),('ME','Maine'),('MD','Maryland'),
 ('MA','Massachusetts'),('MI','Michigan'),('MN','Minnesota'),('MS','Mississippi'),('MO','Missouri'),
 ('MT','Montana'),('NE','Nebraska'),('NV','Nevada'),('NH','New Hampshire'),('NJ','New Jersey'),
 ('NM','New Mexico'),('NY','New York'),('NC','North Carolina'),('ND','North Dakota'),('OH','Ohio'),
 ('OK','Oklahoma'),('OR','Oregon'),('PA','Pennsylvania'),('RI','Rhode Island'),('SC','South Carolina'),
 ('SD','South Dakota'),('TN','Tennessee'),('TX','Texas'),('UT','Utah'),('VT','Vermont'),
 ('VA','Virginia'),('WA','Washington'),('WV','West Virginia'),('WI','Wisconsin'),('WY','Wyoming'),
 ('DC','District of Columbia'),('PR','Puerto Rico'),('VI','U.S. Virgin Islands'),('GU','Guam'),
 ('AS','American Samoa'),('MP','Northern Mariana Islands');
comment on table {SCHEMA}.ref_state is 'USPS code <-> full name lookup, used to normalise the mixed-format state column.';
''', note='Created and populated kyc.ref_state')

print('ref_state rows:', scalar(f'select count(*) from {SCHEMA}.ref_state'))

Created and populated kyc.ref_state  (0.01s)
ref_state rows: 56


## 4 — Load the CSV

`COPY ... FROM STDIN` streamed from the client in 4 MB chunks.

Client-side streaming is not a stylistic choice here — it is the only option. The server runs inside
a Docker container and has no view of `/Users/rudransh/...`, so server-side
`COPY ... FROM '<path>'` could not see the file regardless of privileges.

**Null handling:** in `FORMAT csv`, PostgreSQL's default NULL marker is an *unquoted empty field*, so
the empty strings in this file land as true SQL `NULL`s — which matches the brief's "a null value
means the check did not run".

**Quoting:** `FORMAT csv` applies RFC-4180 rules, so the `reviewer_comment` values containing commas
and embedded newlines are reassembled correctly rather than splitting into extra rows. The row-count
gate in the next cell is what proves this actually happened.

In [11]:
COPY_SQL = f'''
copy {SCHEMA}.kyc_raw (
    fabricated_uid, fabricated_email, fabricated_ssn, enrolled_date, dob, state,
    onboarded_bank_partner, checking_bank_partner, kyc_source, idology_result,
    lexis_nexis_result, persona_idv_result, persona_ssn_result, acro_result,
    manual_review_result, reviewer_comment, overall_kyc_status
)
from stdin with (format csv, header true)
'''

run(f'truncate table {SCHEMA}.kyc_raw;')

t0 = time.time()
CHUNK = 4 * 1024 * 1024
sent = 0
total = CSV_PATH.stat().st_size
with connect() as c, c.cursor() as cur:
    with cur.copy(COPY_SQL) as cp, open(CSV_PATH, 'rb') as f:
        while True:
            buf = f.read(CHUNK)
            if not buf:
                break
            cp.write(buf)
            sent += len(buf)
            pct = sent / total * 100
            print(f'\r  loading {sent/1e6:7.1f} / {total/1e6:.1f} MB  ({pct:5.1f}%)', end='', flush=True)
load_secs = time.time() - t0

raw_rows = scalar(f'select count(*) from {SCHEMA}.kyc_raw')
RUN_LOG.append(('Loaded CSV into kyc.kyc_raw', f'{raw_rows:,} rows', load_secs))
print(f'\nCOPY complete: {raw_rows:,} rows in {load_secs:.1f}s ({raw_rows/load_secs:,.0f} rows/s)')

  loading     4.2 / 273.9 MB  (  1.5%)

  loading     8.4 / 273.9 MB  (  3.1%)

  loading    12.6 / 273.9 MB  (  4.6%)

  loading    16.8 / 273.9 MB  (  6.1%)

  loading    21.0 / 273.9 MB  (  7.7%)

...

  loading   264.2 / 273.9 MB  ( 96.5%)

  loading   268.4 / 273.9 MB  ( 98.0%)

  loading   272.6 / 273.9 MB  ( 99.5%)

  loading   273.9 / 273.9 MB  (100.0%)


COPY complete: 2,515,262 rows in 2.0s (1,251,532 rows/s)


In [12]:
# RECONCILIATION GATE — the loaded row count must equal the independently parsed CSV record count.
print(f'CSV records parsed in Python : {CSV_RECORD_COUNT:,}')
print(f'Rows loaded into kyc_raw     : {raw_rows:,}')
assert raw_rows == CSV_RECORD_COUNT, f'ROW COUNT MISMATCH: {raw_rows} loaded vs {CSV_RECORD_COUNT} parsed'
print('MATCH — no rows lost or duplicated in the load.')

print('\nSample of loaded rows:')
print(q(f'select fabricated_uid, enrolled_date, dob, state, kyc_source, idology_result, overall_kyc_status '
        f'from {SCHEMA}.kyc_raw limit 5').to_string(index=False))

CSV records parsed in Python : 2,515,262
Rows loaded into kyc_raw     : 2,515,262
MATCH — no rows lost or duplicated in the load.

Sample of loaded rows:
  fabricated_uid enrolled_date     dob state kyc_source idology_result overall_kyc_status
DCA9D6C851149610    2026-05-14 07/1996    TX    IDOLOGY           PASS               true
83E70A6D8927CF00    2025-04-01 10/2004    LA    IDOLOGY           PASS               true
F8D8EDC071A1941E    2025-09-15 04/1955    FL    IDOLOGY           PASS               true
DA4F1F9D9CA2C049    2024-06-21 06/1988    FL    IDOLOGY           PASS               true
51D22335D87C7C0E    2026-04-27 12/1993    IN    IDOLOGY           PASS               true


## 5 — Build the typed analytical table

Transformations applied here, each one an explicit, reviewable decision:

| # | Transformation | Reason |
|---|---|---|
| 1 | `overall_kyc_status` `'true'/'false'` → `boolean` | Stored as text in the CSV; boolean makes rate maths safe. |
| 2 | `enrolled_date` → `date` | Uniform `YYYY-MM-DD`, no bad values found. |
| 3 | `dob` `MM/YYYY` → `dob_year`, `dob_month`, `dob_month_start` (date) | Only month precision exists; a real `date` would imply precision we do not have. |
| 4 | `state` → `state_code` via `ref_state` | Column mixes `OH` and `Ohio`; unnormalised it double-counts states. `state_raw` is kept. |
| 5 | Result columns → upper-cased, whitespace-trimmed | Guards against case/padding drift. |
| 6 | `kyc_source = 'Test_user'` → `is_test_user` flag | Test accounts. Flagged, **not deleted**, so denominators stay a deliberate choice. |
| 7 | `ssn_last4` derived from the `*****NNNN` mask | Convenience; the mask is kept verbatim too. |
| 8 | `age_at_enrollment` derived | Useful cut for diagnosing failures; approximate (month precision). |
| 9 | `checks_run` count | Number of provider columns that fired — the basis for waterfall-depth analysis. |

Rows are **never** dropped: `kyc_users` has exactly as many rows as `kyc_raw`.

> **Watch the NULLs.** `kyc_source` is NULL for 7.84% of rows, so `is_test_user` must be built with
> `coalesce(..., false)`. Written as a bare `kyc_source = 'Test_user'`, it evaluates to NULL on those
> rows, and a later `WHERE NOT is_test_user` would silently discard 197,314 users — the very group
> that accounts for ~99% of all non-verifications. This was a live bug in the first run of this
> notebook, caught by the smoke test in §8 showing two populations with identical counts. The columns
> are now `NOT NULL`, so the same mistake would fail the load rather than skew the answer.

In [13]:
run(f'''
drop table if exists {SCHEMA}.kyc_users;
create table {SCHEMA}.kyc_users as
with src as (
    select
        nullif(btrim(fabricated_uid), '')         as user_id,
        nullif(btrim(fabricated_email), '')       as email_token,
        nullif(btrim(fabricated_ssn), '')         as ssn_masked,
        nullif(btrim(enrolled_date), '')          as enrolled_date,
        nullif(btrim(dob), '')                    as dob,
        nullif(btrim(state), '')                  as state_raw,
        nullif(btrim(onboarded_bank_partner), '') as onboarded_bank_partner,
        nullif(btrim(checking_bank_partner), '')  as checking_bank_partner,
        nullif(btrim(kyc_source), '')             as kyc_source,
        upper(nullif(btrim(idology_result), ''))       as idology_result,
        upper(nullif(btrim(lexis_nexis_result), ''))   as lexis_nexis_result,
        upper(nullif(btrim(persona_idv_result), ''))   as persona_idv_result,
        upper(nullif(btrim(persona_ssn_result), ''))   as persona_ssn_result,
        upper(nullif(btrim(acro_result), ''))          as acro_result,
        upper(nullif(btrim(manual_review_result), '')) as manual_review_result,
        nullif(btrim(reviewer_comment), '')       as reviewer_comment,
        lower(nullif(btrim(overall_kyc_status), '')) as overall_kyc_status
    from {SCHEMA}.kyc_raw
)
select
    s.user_id,
    s.email_token,
    s.ssn_masked,
    right(s.ssn_masked, 4)                        as ssn_last4,
    s.enrolled_date::date                         as enrolled_date,
    s.dob                                         as dob_raw,
    split_part(s.dob, '/', 2)::int                as dob_year,
    split_part(s.dob, '/', 1)::int                as dob_month,
    to_date(s.dob, 'MM/YYYY')                     as dob_month_start,
    (extract(year from age(s.enrolled_date::date, to_date(s.dob, 'MM/YYYY'))))::int
                                                  as age_at_enrollment,
    s.state_raw,
    coalesce(rc.state_code, rn.state_code)        as state_code,
    (s.state_raw is not null and coalesce(rc.state_code, rn.state_code) is null)
                                                  as state_unmapped,
    s.onboarded_bank_partner,
    s.checking_bank_partner,
    s.kyc_source,
    s.idology_result,
    s.lexis_nexis_result,
    s.persona_idv_result,
    s.persona_ssn_result,
    s.acro_result,
    s.manual_review_result,
    s.reviewer_comment,
    (s.reviewer_comment is not null)              as has_reviewer_comment,
    (s.overall_kyc_status = 'true')               as is_verified,
    s.overall_kyc_status                          as overall_kyc_status_raw,
    -- The coalesce here is load-bearing. kyc_source is NULL for ~7.8% of rows, so a bare
    -- (kyc_source = 'Test_user') evaluates to NULL -- not false -- on those rows, and
    -- `where not is_test_user` would then silently discard all 197,314 of them alongside
    -- the 107 real test users. That is the single largest group of non-verifications, so
    -- the three-valued-logic trap would have quietly changed the headline number.
    coalesce(s.kyc_source = 'Test_user', false)   as is_test_user,
    (  (s.idology_result       is not null)::int
     + (s.lexis_nexis_result   is not null)::int
     + (s.persona_idv_result   is not null)::int
     + (s.persona_ssn_result   is not null)::int
     + (s.acro_result          is not null)::int
     + (s.manual_review_result is not null)::int )  as checks_run
from src s
left join {SCHEMA}.ref_state rc on upper(s.state_raw) = rc.state_code
left join {SCHEMA}.ref_state rn on lower(s.state_raw) = lower(rn.state_name);
''', note='Built kyc.kyc_users (typed analytical table)')

print('kyc_users rows:', f"{scalar(f'select count(*) from {SCHEMA}.kyc_users'):,}")

# Guard the fix: every boolean flag must be strictly TRUE/FALSE, never NULL. If any of these
# is non-zero, a filter like `where not <flag>` would silently drop rows.
print('\nNULLs in boolean flag columns (must all be 0):')
print(q(f'''
    select count(*) filter (where is_verified          is null) as is_verified,
           count(*) filter (where is_test_user         is null) as is_test_user,
           count(*) filter (where state_unmapped       is null) as state_unmapped,
           count(*) filter (where has_reviewer_comment is null) as has_reviewer_comment
      from {SCHEMA}.kyc_users
''').to_string(index=False))

Built kyc.kyc_users (typed analytical table)  (3.57s)


kyc_users rows: 2,515,262

NULLs in boolean flag columns (must all be 0):
 is_verified  is_test_user  state_unmapped  has_reviewer_comment
           0             0               0                     0


In [14]:
# Constraints and indexes. Added after the bulk build — far faster than maintaining them during it.
# NOT NULL on the boolean flags is a real safeguard, not decoration: it makes the three-valued-logic
# bug described in the previous cell impossible to reintroduce, because the load would fail instead.
run(f'''
alter table {SCHEMA}.kyc_users
    alter column user_id set not null,
    alter column enrolled_date set not null,
    alter column is_verified set not null,
    alter column is_test_user set not null,
    alter column state_unmapped set not null,
    alter column has_reviewer_comment set not null,
    alter column checks_run set not null,
    add constraint pk_kyc_users primary key (user_id);

alter table {SCHEMA}.kyc_users
    add constraint ck_idology  check (idology_result       in ('PASS','FAIL')) not valid,
    add constraint ck_lexis    check (lexis_nexis_result   in ('PASS','FAIL')) not valid,
    add constraint ck_pidv     check (persona_idv_result   in ('PASS'))        not valid,
    add constraint ck_pssn     check (persona_ssn_result   in ('PASS'))        not valid,
    add constraint ck_acro     check (acro_result          in ('PASS','FAIL')) not valid,
    add constraint ck_manual   check (manual_review_result in ('PASS','FAIL')) not valid,
    add constraint ck_checks   check (checks_run between 0 and 6)              not valid;
''', note='Added primary key, NOT NULLs and CHECK constraints to kyc.kyc_users')

# VALIDATE turns the NOT VALID constraints into a real assertion over every existing row.
# If any of these raise, the data contradicts the brief's stated domains.
for ck in ['ck_idology','ck_lexis','ck_pidv','ck_pssn','ck_acro','ck_manual','ck_checks']:
    run(f'alter table {SCHEMA}.kyc_users validate constraint {ck};')
print('All CHECK constraints validated against the loaded data — provider result domains confirmed.')

run(f'''
create index ix_kyc_users_verified   on {SCHEMA}.kyc_users (is_verified);
create index ix_kyc_users_source     on {SCHEMA}.kyc_users (kyc_source);
create index ix_kyc_users_enrolled   on {SCHEMA}.kyc_users (enrolled_date);
create index ix_kyc_users_state      on {SCHEMA}.kyc_users (state_code);
create index ix_kyc_users_idology    on {SCHEMA}.kyc_users (idology_result);
create index ix_kyc_users_partner    on {SCHEMA}.kyc_users (onboarded_bank_partner);
create index ix_kyc_users_checks     on {SCHEMA}.kyc_users (checks_run);
create index ix_kyc_users_testuser   on {SCHEMA}.kyc_users (is_test_user) where is_test_user;
''', note='Created indexes on kyc.kyc_users')

run(f'analyze {SCHEMA}.kyc_users;', note='ANALYZE kyc.kyc_users')
run(f'analyze {SCHEMA}.kyc_raw;')

Added primary key, NOT NULLs and CHECK constraints to kyc.kyc_users  (2.04s)


All CHECK constraints validated against the loaded data — provider result domains confirmed.


Created indexes on kyc.kyc_users  (3.68s)


ANALYZE kyc.kyc_users  (0.26s)


0.2168588638305664

## 6 — Grants

`kyc_ro` gets read-only, `kyc_rw` gets DML, `kyc_owner` gets everything. Default privileges are set so
tables created later in this schema inherit the same grants automatically. All three are then granted
to the connecting user.

In [15]:
run(f'''
grant usage on schema {SCHEMA} to kyc_ro, kyc_rw, kyc_owner;
grant create on schema {SCHEMA} to kyc_owner;

grant select on all tables in schema {SCHEMA} to kyc_ro;
grant select, insert, update, delete on all tables in schema {SCHEMA} to kyc_rw;
grant all privileges on all tables in schema {SCHEMA} to kyc_owner;
grant usage, select on all sequences in schema {SCHEMA} to kyc_rw, kyc_owner;

-- future tables in this schema inherit the same grants
alter default privileges in schema {SCHEMA} grant select on tables to kyc_ro;
alter default privileges in schema {SCHEMA} grant select, insert, update, delete on tables to kyc_rw;
alter default privileges in schema {SCHEMA} grant all privileges on tables to kyc_owner;
''', note='Granted privileges on schema kyc to kyc_ro / kyc_rw / kyc_owner')

# Give the connecting user membership of all three group roles.
run(f"grant kyc_owner, kyc_rw, kyc_ro to {CONN['user']};",
    note=f"Granted kyc_owner/kyc_rw/kyc_ro to {CONN['user']}")

print(q(f'''
    select grantee, table_name, string_agg(distinct privilege_type, ', ' order by privilege_type) as privileges
      from information_schema.role_table_grants
     where table_schema = '{SCHEMA}' and grantee like 'kyc%'
     group by grantee, table_name
     order by table_name, grantee
''').to_string(index=False))

Granted privileges on schema kyc to kyc_ro / kyc_rw / kyc_owner  (0.01s)
Granted kyc_owner/kyc_rw/kyc_ro to postgres  (0.00s)
  grantee table_name                                                    privileges
kyc_owner    kyc_raw DELETE, INSERT, REFERENCES, SELECT, TRIGGER, TRUNCATE, UPDATE
   kyc_ro    kyc_raw                                                        SELECT
   kyc_rw    kyc_raw                                DELETE, INSERT, SELECT, UPDATE
kyc_owner  kyc_users DELETE, INSERT, REFERENCES, SELECT, TRIGGER, TRUNCATE, UPDATE
   kyc_ro  kyc_users                                                        SELECT
   kyc_rw  kyc_users                                DELETE, INSERT, SELECT, UPDATE
kyc_owner  ref_state DELETE, INSERT, REFERENCES, SELECT, TRIGGER, TRUNCATE, UPDATE
   kyc_ro  ref_state                                                        SELECT
   kyc_rw  ref_state                                DELETE, INSERT, SELECT, UPDATE


## 7 — Post-load validation

Checks that must all pass before the table is trusted for analysis.

In [16]:
checks = []

raw_n   = scalar(f'select count(*) from {SCHEMA}.kyc_raw')
typed_n = scalar(f'select count(*) from {SCHEMA}.kyc_users')
checks.append(('Row count: CSV parse == kyc_raw', CSV_RECORD_COUNT == raw_n, f'{CSV_RECORD_COUNT:,} vs {raw_n:,}'))
checks.append(('Row count: kyc_raw == kyc_users (no rows dropped)', raw_n == typed_n, f'{raw_n:,} vs {typed_n:,}'))

dups = scalar(f'select count(*) from (select user_id from {SCHEMA}.kyc_users group by 1 having count(*) > 1) t')
checks.append(('user_id is unique', dups == 0, f'{dups} duplicates'))

nulls = scalar(f'select count(*) from {SCHEMA}.kyc_users where user_id is null or enrolled_date is null or is_verified is null')
checks.append(('No NULLs in key columns', nulls == 0, f'{nulls} rows'))

# Guards the three-valued-logic bug: a NULL boolean makes `where not <flag>` drop rows silently.
bool_nulls = scalar(f'''select count(*) from {SCHEMA}.kyc_users
                         where is_verified is null or is_test_user is null
                            or state_unmapped is null or has_reviewer_comment is null''')
checks.append(('No NULLs in any boolean flag (3VL guard)', bool_nulls == 0, f'{bool_nulls} rows'))

unmapped = scalar(f'select count(*) from {SCHEMA}.kyc_users where state_unmapped')
checks.append(('Every non-null state mapped to a USPS code', unmapped == 0, f'{unmapped} unmapped'))

bad_status = scalar(f"select count(*) from {SCHEMA}.kyc_users where overall_kyc_status_raw not in ('true','false')")
checks.append(('overall_kyc_status only ever true/false', bad_status == 0, f'{bad_status} other values'))

# Verified count must survive the text -> boolean conversion unchanged.
raw_true   = scalar(f"select count(*) from {SCHEMA}.kyc_raw where overall_kyc_status = 'true'")
typed_true = scalar(f'select count(*) from {SCHEMA}.kyc_users where is_verified')
checks.append(('Verified count preserved through boolean cast', raw_true == typed_true, f'{raw_true:,} vs {typed_true:,}'))

# Test-user flag must match a direct count against the raw layer.
raw_test   = scalar(f"select count(*) from {SCHEMA}.kyc_raw where kyc_source = 'Test_user'")
typed_test = scalar(f'select count(*) from {SCHEMA}.kyc_users where is_test_user')
checks.append(('is_test_user matches raw Test_user count', raw_test == typed_test, f'{raw_test} vs {typed_test}'))

# Partitioning check: the three populations must sum back to the whole.
part = scalar(f'''select count(*) from {SCHEMA}.kyc_users
                   where not (is_test_user or (not is_test_user and kyc_source is null)
                                           or (not is_test_user and kyc_source is not null))''')
checks.append(('Test / no-source / sourced populations partition cleanly', part == 0, f'{part} unaccounted'))

date_rng = q(f'select min(enrolled_date) as min_d, max(enrolled_date) as max_d from {SCHEMA}.kyc_users')
checks.append(('enrolled_date parsed to a sane range', True, f"{date_rng.min_d[0]} .. {date_rng.max_d[0]}"))

print(f"{'CHECK':56s} {'RESULT':10s} DETAIL")
print('-' * 104)
for name, ok, detail in checks:
    print(f'{name:56s} {"PASS" if ok else "** FAIL **":10s} {detail}')

VALIDATION_OK = all(ok for _, ok, _ in checks)
print('-' * 104)
print('OVERALL:', 'ALL CHECKS PASSED' if VALIDATION_OK else '*** FAILURES PRESENT — DO NOT ANALYSE YET ***')
assert VALIDATION_OK, 'Post-load validation failed — see the table above.'

CHECK                                                    RESULT     DETAIL
--------------------------------------------------------------------------------------------------------
Row count: CSV parse == kyc_raw                          PASS       2,515,262 vs 2,515,262
Row count: kyc_raw == kyc_users (no rows dropped)        PASS       2,515,262 vs 2,515,262
user_id is unique                                        PASS       0 duplicates
No NULLs in key columns                                  PASS       0 rows
No NULLs in any boolean flag (3VL guard)                 PASS       0 rows
Every non-null state mapped to a USPS code               PASS       0 unmapped
overall_kyc_status only ever true/false                  PASS       0 other values
Verified count preserved through boolean cast            PASS       2,315,747 vs 2,315,747
is_test_user matches raw Test_user count                 PASS       107 vs 107
Test / no-source / sourced populations partition cleanly PASS       0 unacc

In [17]:
# Column-by-column null counts in the typed table, next to the CSV profile, as a final cross-check.
db_nulls = q(f'''
    select 'idology_result' as col, count(*) filter (where idology_result is null) as nulls from {SCHEMA}.kyc_users
    union all select 'lexis_nexis_result',   count(*) filter (where lexis_nexis_result is null)   from {SCHEMA}.kyc_users
    union all select 'persona_idv_result',   count(*) filter (where persona_idv_result is null)   from {SCHEMA}.kyc_users
    union all select 'persona_ssn_result',   count(*) filter (where persona_ssn_result is null)   from {SCHEMA}.kyc_users
    union all select 'acro_result',          count(*) filter (where acro_result is null)          from {SCHEMA}.kyc_users
    union all select 'manual_review_result', count(*) filter (where manual_review_result is null) from {SCHEMA}.kyc_users
    union all select 'kyc_source',           count(*) filter (where kyc_source is null)           from {SCHEMA}.kyc_users
    union all select 'dob_raw',              count(*) filter (where dob_raw is null)              from {SCHEMA}.kyc_users
    union all select 'state_raw',            count(*) filter (where state_raw is null)            from {SCHEMA}.kyc_users
''')
db_nulls['csv_empty'] = db_nulls['col'].map(lambda c: empties[{'dob_raw':'dob','state_raw':'state'}.get(c, c)])
db_nulls['match'] = db_nulls['nulls'] == db_nulls['csv_empty']
print(db_nulls.to_string(index=False))
print('\nAll null counts reconcile:', bool(db_nulls['match'].all()))

                 col   nulls  csv_empty  match
      idology_result   94184      94184   True
           state_raw     878        878   True
             dob_raw     868        868   True
          kyc_source  197314     197314   True
  persona_idv_result 2512667    2512667   True
         acro_result 2401035    2401035   True
  lexis_nexis_result 2238263    2238263   True
  persona_ssn_result 2504529    2504529   True
manual_review_result 2440834    2440834   True

All null counts reconcile: True


## 8 — Smoke-test the table, and settle the one question that decides the headline

Not the Task 1 analysis — but enough to prove the table answers questions correctly, and to
resolve the ambiguity that would otherwise sit under every number produced from it.

The dataset's non-verifications are dominated by a single group: 197,314 users (7.84%) with a null
`kyc_source`, of whom 197,298 are unverified — **98.9% of every non-verification in the file**. Whether
those are genuine rejections or a data artefact decides the headline rate, so it is settled here with
evidence rather than left as an assumption.

In [18]:
print('Non-verification rate, on four candidate denominators:\n')
print(q(f'''
    with pops as (
        select 1 as ord, 'All rows'                                     as population, * from {SCHEMA}.kyc_users
        union all
        select 2, 'Excluding test users',                               * from {SCHEMA}.kyc_users where not is_test_user
        union all
        select 3, 'Excl. test users AND rows with no kyc_source',       * from {SCHEMA}.kyc_users where not is_test_user and kyc_source is not null
        union all
        select 4, 'Excl. test users AND rows where no check ever ran',  * from {SCHEMA}.kyc_users where not is_test_user and checks_run > 0
    )
    select population,
           count(*)                                    as users,
           count(*) filter (where not is_verified)     as not_verified,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct_not_verified
      from pops
     group by ord, population
     order by ord
''').to_string(index=False))

print('''
Reading this table:
  Row 1  The honest all-in number, and the one to lead with: 7.93% against the brief's 5% bar.
  Row 2  Barely moves it -- the 107 test users are immaterial.
  Row 3  Looks dramatic (0.09%) but is CIRCULAR, and must not be used as a headline.
         kyc_source names the provider that produced a PASS, so it is null precisely when
         nobody passed. Excluding those rows excludes almost every rejection by construction.
  Row 4  The defensible version of the same idea: drop only users where genuinely no check
         ran (843 of them). The rate hardly moves -- 7.90%.

  Conclusion: the rejections are real, not a reporting artefact. So the question worth asking
  is not "are these rows valid?" but "were these users given a fair run through the waterfall?"
  -- which the next cell addresses.''')

Non-verification rate, on four candidate denominators:



                                       population   users  not_verified pct_not_verified
                                         All rows 2515262        199515             7.93
                             Excluding test users 2515155        199421             7.93
     Excl. test users AND rows with no kyc_source 2317841          2123             0.09
Excl. test users AND rows where no check ever ran 2514312        198578             7.90

Reading this table:
  Row 1  The honest all-in number, and the one to lead with: 7.93% against the brief's 5% bar.
  Row 2  Barely moves it -- the 107 test users are immaterial.
  Row 3  Looks dramatic (0.09%) but is CIRCULAR, and must not be used as a headline.
         kyc_source names the provider that produced a PASS, so it is null precisely when
         nobody passed. Excluding those rows excludes almost every rejection by construction.
  Row 4  The defensible version of the same idea: drop only users where genuinely no check
         ran (84

In [19]:
# What actually happened to the users with no kyc_source? This is the group that decides the
# headline, so characterise it rather than assuming. Two hypotheses were on the table:
#   (a) a logging gap  -- checks ran, but kyc_source was never written back;
#   (b) waterfall exhaustion -- checks ran and all of them failed.
# The result-pattern breakdown below settles it.
print('Exact provider-result pattern for users with kyc_source IS NULL:\n')
print(q(f'''
    select coalesce(idology_result,'-')       as idology,
           coalesce(lexis_nexis_result,'-')   as lexis,
           coalesce(persona_idv_result,'-')   as p_idv,
           coalesce(persona_ssn_result,'-')   as p_ssn,
           coalesce(acro_result,'-')          as acro,
           coalesce(manual_review_result,'-') as manual,
           checks_run,
           count(*)                           as users,
           count(*) filter (where is_verified) as verified
      from {SCHEMA}.kyc_users
     where kyc_source is null
     group by 1,2,3,4,5,6,7
     order by users desc
''').to_string(index=False))

any_pass = scalar(f'''
    select count(*) from {SCHEMA}.kyc_users
     where kyc_source is null
       and 'PASS' in (idology_result, lexis_nexis_result, persona_idv_result,
                      persona_ssn_result, acro_result, manual_review_result)''')
print(f'\nUsers with kyc_source IS NULL that have ANY provider PASS: {any_pass}')

print('''
Verdict: hypothesis (b). Not one of these users has a single PASS on any provider -- every
recorded result is FAIL. kyc_source is therefore not missing data; it is null by design when
no provider cleared the user. These are real non-verifications.

But note the largest pattern in the table above: users who FAILED Idology and then had
NOTHING else run at all. They were never routed to the secondary providers the brief
describes. That is not a genuine rejection -- it is an avoidable loss to a routing gap,
and it is the single biggest bucket in the whole dataset.''')

print('\nSplitting non-verifications into genuine rejections vs avoidable process loss:\n')
print(q(f'''
    with nv as (select * from {SCHEMA}.kyc_users where not is_verified and not is_test_user)
    select case
             when checks_run = 0                        then '1. No check ever ran'
             when idology_result = 'FAIL'
              and checks_run = 1                        then '2. Failed Idology, never routed onward'
             else                                            '3. Ran 2+ checks, failed them all'
           end as bucket,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct_of_non_verified
      from nv
     group by 1 order by 1
''').to_string(index=False))
print('''
Buckets 1 and 2 are process losses -- users the waterfall never finished running.
Bucket 3 is where genuine rejection actually lives. Quantifying and attacking bucket 2 is
the highest-leverage finding available for Task 1.''')

Exact provider-result pattern for users with kyc_source IS NULL:

idology lexis p_idv p_ssn acro manual  checks_run  users  verified
   FAIL     -     -     -    -      -           1 101162         3
   FAIL     -     -     - FAIL   FAIL           3  27597         2
   FAIL     -     -     - FAIL      -           2  24922         1
   FAIL  FAIL     -     -    -      -           2  21449         2
   FAIL     -     -     -    -   FAIL           2  14571         1
   FAIL  FAIL     -     -    -   FAIL           3   3278         4
   FAIL  FAIL     -     - FAIL   FAIL           4   2391         3
   FAIL  FAIL     -     - FAIL      -           3   1099         0
      -     -     -     -    -      -           0    843         0
      -  FAIL     -     -    -      -           1      2         0

Users with kyc_source IS NULL that have ANY provider PASS: 0

Verdict: hypothesis (b). Not one of these users has a single PASS on any provider -- every
recorded result is FAIL. kyc_source is ther

In [20]:
print('Outcome by kyc_source (the column that records which check produced the result):\n')
print(q(f'''
    select coalesce(kyc_source, '<NULL>') as kyc_source,
           count(*)                                        as users,
           count(*) filter (where is_verified)              as verified,
           count(*) filter (where not is_verified)          as not_verified,
           round(100.0 * count(*) filter (where not is_verified) / count(*), 2) as pct_not_verified
      from {SCHEMA}.kyc_users
     group by 1
     order by users desc
''').to_string(index=False))

Outcome by kyc_source (the column that records which check produced the result):



           kyc_source   users  verified  not_verified pct_not_verified
              IDOLOGY 2078101   2077708           393             0.02
               <NULL>  197314        16        197298            99.99
ProviderA_Lexis_Nexis  149364    148690           674             0.45
                 ACRO   44563     44297           266             0.60
        MANUAL_REVIEW   22156     22124            32             0.14
ProviderB_Lexis_Nexis   15998     15752           246             1.54
          PERSONA_SSN    5146      4995           151             2.93
          PERSONA_IDV    2513      2152           361            14.37
            Test_user     107        13            94            87.85


In [21]:
print('Waterfall depth — how many provider checks actually ran per user:\n')
print(q(f'''
    select checks_run,
           count(*) as users,
           round(100.0 * count(*) / sum(count(*)) over (), 2) as pct_of_all,
           count(*) filter (where is_verified) as verified,
           round(100.0 * count(*) filter (where is_verified) / count(*), 2) as pct_verified
      from {SCHEMA}.kyc_users
     group by 1 order by 1
''').to_string(index=False))

print('\nTable sizes on disk:')
print(q(f'''
    select relname as table_name,
           to_char(n_live_tup, 'FM9,999,999,999') as approx_rows,
           pg_size_pretty(pg_total_relation_size(relid)) as total_size
      from pg_stat_user_tables where schemaname = '{SCHEMA}' order by pg_total_relation_size(relid) desc
''').to_string(index=False))

Waterfall depth — how many provider checks actually ran per user:



 checks_run   users pct_of_all  verified pct_verified
          0     843       0.03         0         0.00
          1 2195948      87.30   2094593        95.38
          2  255718      10.17    193866        75.81
          3   58338       2.32     25481        43.68
          4    4413       0.18      1805        40.90
          5       2       0.00         2       100.00

Table sizes on disk:
table_name approx_rows total_size
 kyc_users   2,515,339     624 MB
   kyc_raw   2,515,304     347 MB
 ref_state          56      48 kB


## 9 — Append this run to the audit log

Writes a timestamped, machine-generated record of *this* execution to `AUDIT_LOG.md`. The narrative
sections of that file (assumptions, decisions, caveats) are maintained by hand; this cell only ever
appends below the run-history marker.

In [22]:
MARKER = '<!-- RUN-HISTORY: appended automatically by notebooks/01_kyc_load_and_setup.ipynb -->'
stamp = dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')
server = scalar('select version()').split(',')[0]

lines = [
    '',
    f'### Run — {stamp}',
    '',
    f'- **Server:** {server}',
    f"- **Target:** `{CONN['user']}@{CONN['host']}:{CONN['port']}/{CONN['dbname']}`, schema `{SCHEMA}`",
    f'- **Source file:** `{CSV_PATH.name}` ({CSV_PATH.stat().st_size:,} bytes)',
    f'- **Records parsed from CSV:** {CSV_RECORD_COUNT:,}',
    f'- **Rows in `kyc.kyc_raw`:** {raw_n:,}',
    f'- **Rows in `kyc.kyc_users`:** {typed_n:,}',
    f"- **Validation:** {'ALL CHECKS PASSED' if VALIDATION_OK else 'FAILURES PRESENT'}",
    '',
    '| Step | Detail | Seconds |',
    '|---|---|---|',
]
for step, detail, secs in RUN_LOG:
    lines.append(f'| {step} | {detail} | {secs:.2f} |')
lines.append('')

text = AUDIT_MD.read_text() if AUDIT_MD.exists() else MARKER + '\n'
if MARKER not in text:
    text += '\n\n## Run history\n\n' + MARKER + '\n'
text = text.rstrip() + '\n' + '\n'.join(lines)
AUDIT_MD.write_text(text)

print(f'Appended run record to {AUDIT_MD}')
print('\n'.join(lines))

Appended run record to /Users/rudransh/d_drive/GITHUB/Brightmoney/AUDIT_LOG.md

### Run — 2026-09-06 23:49:51 IST

- **Server:** PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on aarch64-unknown-linux-gnu
- **Target:** `postgres@localhost:5432/study`, schema `kyc`
- **Source file:** `KYC_Synthetic_Dataset.csv` (273,899,427 bytes)
- **Records parsed from CSV:** 2,515,262
- **Rows in `kyc.kyc_raw`:** 2,515,262
- **Rows in `kyc.kyc_users`:** 2,515,262
- **Validation:** ALL CHECKS PASSED

| Step | Detail | Seconds |
|---|---|---|
| Profiled CSV | 2,515,262 records, 17 columns | 9.55 |
| Created roles kyc_owner / kyc_rw / kyc_ro |  | 0.01 |
| Created schema kyc |  | 0.01 |
| Created kyc.kyc_raw (landing table) |  | 0.03 |
| Created and populated kyc.ref_state |  | 0.01 |
| Loaded CSV into kyc.kyc_raw | 2,515,262 rows | 2.01 |
| Built kyc.kyc_users (typed analytical table) |  | 3.57 |
| Added primary key, NOT NULLs and CHECK constraints to kyc.kyc_users |  | 2.04 |
| Created indexes on kyc.kyc_use